# 🛡️ Trust Before Answer: Ethical AI Guardrail Agent

## Project Summary

This project implements a **"Trust Before Answer"** system — an AI agent that incorporates ethical guardrails to **prevent hallucination** and ensure **knowledge sufficiency** before generating responses. The system uses a **Retrieval-Augmented Generation (RAG)** pipeline combined with a guardrail mechanism to guarantee that every answer is grounded in verified knowledge.

**Knowledge Source**: [help.webex.com](https://help.webex.com) — Cisco Webex documentation covering meetings, messaging, security, and more.

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Hallucination Prevention** | The system refuses to answer when it lacks sufficient context, preventing the LLM from fabricating information |
| **Knowledge Sufficiency Check** | Before generating any answer, the system verifies that relevant context exists in its Webex knowledge base |
| **Ethical AI Guardrail** | A prompt-engineered constraint that forces the model to only answer using provided evidence |
| **RAG Pipeline** | Retrieval-Augmented Generation ensures answers are grounded in actual Webex documentation |

### Architecture Flow

```
User Query → [Retrieval from Webex KB] → [Knowledge Sufficiency Check] → [Guardrail LLM Generation] → Response
                                              ↓ (if no context found)
                                        REFUSAL (safe denial)
```

### Models & Technologies Used

- **LLM**: Google Flan-T5-Base (runs locally, **NO API key required**)
- **Knowledge Source**: help.webex.com (Cisco Webex documentation)
- **Framework**: LangChain (prompt templates, output parsers, chain composition)
- **UI**: Gradio (interactive web interface)
- **Paradigm**: Retrieval-Augmented Generation with ethical constraints

---

## Block 1: Package Installation

This block installs all the required Python packages:

- **`transformers`**: Hugging Face library for loading pre-trained models (Flan-T5)
- **`torch`**: PyTorch deep learning framework (model inference backend)
- **`langchain-huggingface`**: LangChain integration with Hugging Face models
- **`langchain-core`**: Core LangChain components (prompts, parsers, chains)
- **`gradio`**: Web UI framework for building interactive ML demos

**No API key is needed** — all models run locally on your machine.

In [43]:
# Block 1: Install required packages
# All models run locally — NO API key needed!

import subprocess
import sys

packages = [
    "transformers",
    "torch",
    "langchain-huggingface",
    "langchain-core",
    "langchain-text-splitters",
    "PyPDF2",
    "gradio>=4.44.0",
    "gradio_client>=1.3.0"
]

for pkg in packages:
    print(f"Installing {pkg}...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )

print("\n✅ All packages installed successfully!")
print("   No API key required — models run locally.")

Installing transformers...
Installing torch...
Installing langchain-huggingface...
Installing langchain-core...
Installing langchain-text-splitters...
Installing PyPDF2...
Installing gradio>=4.44.0...
Installing gradio_client>=1.3.0...

✅ All packages installed successfully!
   No API key required — models run locally.


## Block 2: Import Libraries

This block imports all necessary modules:

| Library | Purpose |
|---------|--------|
| `os` | File path operations |
| `warnings` | Suppress noisy warnings from transformers |
| `gradio` | Building the interactive web UI |
| `transformers` | Loading and running the Flan-T5 model locally |
| `HuggingFacePipeline` | LangChain wrapper to use HF models in chains |
| `PromptTemplate` | Structured prompt engineering with variables |
| `StrOutputParser` | Parses LLM output into plain strings |

**No API key setup needed!** Everything runs on your local machine.

In [44]:
# Block 2: Import libraries
# No API key configuration needed!

import os
import warnings
warnings.filterwarnings("ignore")

import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


## Block 3: Initialize the Local Language Model

This block downloads and loads the **Google Flan-T5-Base** model locally:

- **`google/flan-t5-base`**: A 250M parameter instruction-tuned model that runs on any machine
- **Runs entirely offline** after first download (model is cached)
- **No API key, no internet needed** after initial download
- **`do_sample=False`**: Deterministic output for consistent guardrail behavior

### Why Flan-T5?

| Feature | Benefit |
|---------|--------|
| Free & open-source | No costs, no API keys |
| Instruction-tuned | Follows prompt instructions well |
| Small (250M params) | Runs on any laptop without GPU |
| Fast inference | Quick response times |

The first run will download the model (~990MB). Subsequent runs use the cached version.

In [45]:

# Block 3: Initialize the local LLM (Flan-T5)
# Downloads the model on first run, then uses cache
# NO API KEY NEEDED!

MODEL_NAME = "google/flan-t5-base"

print(f"⏳ Loading model: {MODEL_NAME}")
print("   (First run downloads ~990MB, subsequent runs use cache)")

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Create a text2text-generation pipeline
# TUNED: max_new_tokens=512 for fuller answers, num_beams=4 for better quality
pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,
    num_beams=4,
    early_stopping=True,
    no_repeat_ngram_size=3,
)

# Wrap in LangChain for chain composition
llm = HuggingFacePipeline(pipeline=pipe)

print(f"\n✅ Model loaded successfully: {MODEL_NAME}")
print("   - Runs locally (no internet needed after download)")
print("   - No API key required")
print("   - Beam search: 4 beams (better quality)")
print("   - Max tokens: 512 (fuller answers)")
print("   - No-repeat 3-gram (reduces repetition)")


⏳ Loading model: google/flan-t5-base
   (First run downloads ~990MB, subsequent runs use cache)


Device set to use mps:0



✅ Model loaded successfully: google/flan-t5-base
   - Runs locally (no internet needed after download)
   - No API key required
   - Beam search: 4 beams (better quality)
   - Max tokens: 512 (fuller answers)
   - No-repeat 3-gram (reduces repetition)


## Block 4: Knowledge Retrieval Function

This block implements the **retrieval** component of the RAG pipeline. The `retrieve_context()` function simulates searching a knowledge base for information relevant to the user's query.

### How it works:

1. A **mock knowledge base** (dictionary) stores verified facts sourced from [help.webex.com](https://help.webex.com)
2. The function performs **keyword matching** — if the user's query (lowercased) contains a key from the knowledge base, the corresponding fact is returned
3. If **no match is found**, the function returns `None` — this signals to the pipeline that there is insufficient knowledge to answer safely

### Knowledge Base Entries (sourced from help.webex.com):

| Keyword | Topic |
|---------|-------|
| `meeting` | Scheduling and joining Webex meetings |
| `messaging` / `space` | Webex messaging, spaces, and collaboration features |
| `share` / `content` | Sharing content during meetings |
| `noise` / `background` | Noise removal and virtual backgrounds |
| `security` / `encryption` | End-to-end encryption and security features |
| `api` / `bot` | Webex developer APIs and bot accounts |

### Production Note:
In a real deployment, this function would be replaced with a **vector store search** (e.g., FAISS, Pinecone, ChromaDB) that performs semantic similarity matching against embedded documents from help.webex.com.

In [46]:

# Block 4: Knowledge Retrieval Function
# IMPROVED: Best-match keyword selection + partial/stem matching
# Simulates retrieval from a verified knowledge base (sourced from help.webex.com)

import re  # needed for word-boundary matching

def retrieve_context(query):
    """
    Improved retrieval from Webex knowledge base.
    
    Fixes applied:
    1. Partial/stem matching (e.g., "encrypted" matches "encryption")
    2. Best-match selection (most specific keyword wins, not first match)
    3. Multi-keyword fusion (combines context when multiple topics match)
    
    Args:
        query (str): The user's question/input
    
    Returns:
        str or None: Retrieved context if found, None if no relevant info exists
    """
    # Knowledge base with verified facts from help.webex.com
    knowledge_base = {
        "meeting": (
            "In Webex App, you can schedule meetings directly from a space or "
            "from the Meetings tab. You can join meetings with a single tap across "
            "any device — desktop, phone, or car — using the Move to Mobile QR code "
            "feature and Apple CarPlay integration. Webex supports real-time "
            "translation to 100+ languages, closed captioning, gesture recognition "
            "for in-meeting reactions, and Webex Assistant for voice commands, "
            "transcription, and automatic note-taking. Meetings can scale from "
            "small team calls up to webinars with 10,000 attendees."
        ),
        "messaging": (
            "Webex Messaging keeps work flowing between meetings with asynchronous "
            "collaboration. You can create spaces organized by workstreams, "
            "edit, @mention, forward, flag, pin, and thread messages. Spaces support "
            "file sharing up to 100MB per attachment with formats including Word, Excel, "
            "PowerPoint, PDF, JPEG, PNG, GIF, and BMP. You can co-edit documents, "
            "access meeting artifacts, and collaborate with anyone outside your "
            "organization by adding their email to a space. Webex integrates with "
            "100+ third-party apps including Google, Microsoft, Salesforce, and Slack."
        ),
        "space": (
            "Webex Spaces allow you to organize team communication by workstreams. "
            "You can create spaces for specific projects or topics, add internal "
            "and external participants by email, share files, co-edit documents, "
            "and schedule meetings directly within a space. Spaces support threading, "
            "pinning important messages, advanced search with filters, end-to-end "
            "encryption, and two-way whiteboarding for visual collaboration."
        ),
        "share": (
            "In Webex meetings, you can share your entire screen, a specific "
            "application window, or a file directly. Webex supports immersive share "
            "for presentations, allowing your video to appear integrated with your "
            "content. Supported file attachment types include Microsoft Word (doc, docx), "
            "Excel (xls, xlsx), PowerPoint (ppt, pptx), PDF, JPEG, BMP, GIF, and PNG. "
            "Files can be shared via local upload (multipart/form-data) or remote URL. "
            "The maximum file attachment size is 100MB per file."
        ),
        "content": (
            "In Webex meetings, you can share your entire screen, a specific "
            "application window, or a file directly. Webex supports immersive share "
            "for presentations, allowing your video to appear integrated with your "
            "content. Supported file attachment types include Microsoft Word (doc, docx), "
            "Excel (xls, xlsx), PowerPoint (ppt, pptx), PDF, JPEG, BMP, GIF, and PNG. "
            "Files can be shared via local upload (multipart/form-data) or remote URL. "
            "The maximum file attachment size is 100MB per file."
        ),
        "noise": (
            "Webex includes built-in noise removal technology that filters out "
            "background sounds such as keyboard typing, dog barking, or construction "
            "noise, letting you meet with confidence from any location. Additionally, "
            "voice optimization ensures your speech is clear and natural-sounding "
            "regardless of your environment."
        ),
        "background": (
            "Webex App allows you to use a virtual or blurred background in calls "
            "and meetings. You can choose from preset backgrounds, upload your own "
            "custom image, or blur your surroundings for privacy. Webex also features "
            "built-in noise removal technology and people-focused views that frame "
            "participants optimally regardless of their position in the room."
        ),
        "security": (
            "Webex provides end-to-end encryption for messages, files, and meetings. "
            "Organizations can enable anti-malware scanning of files to protect users "
            "from malicious content. Files under evaluation are quarantined and scanned; "
            "infected files become permanently unavailable for download (410 Gone). "
            "Webex requires Server Name Indication (SNI) for TLS/SSL connections and "
            "uses enterprise-grade security managed through Cisco Control Hub."
        ),
        "encryption": (
            "Webex provides end-to-end encryption for messages, files, and meetings. "
            "Organizations can enable anti-malware scanning of files to protect users "
            "from malicious content. Files under evaluation are quarantined and scanned; "
            "infected files become permanently unavailable for download (410 Gone). "
            "Webex requires Server Name Indication (SNI) for TLS/SSL connections and "
            "uses enterprise-grade security managed through Cisco Control Hub."
        ),
        "api": (
            "The Webex REST API supports pagination using RFC5988 Web Linking standard "
            "with Link headers containing rel='next' for navigating pages. The API "
            "enforces rate limits of approximately 300 requests per minute, returning "
            "429 Too Many Requests with a Retry-After header when exceeded. Message "
            "attachments are limited to 100MB each and can be sent via local file upload "
            "(multipart/form-data) or remote URL. Standard HTTP response codes are used: "
            "200 OK, 201 Created, 401 Unauthorized, 429 Too Many Requests, etc."
        ),
        "bot": (
            "Webex bot accounts have less restrictive rate limits than end-user "
            "accounts, making them suitable for automation. Bots can send messages with "
            "attachments, use Markdown formatting (bold, italic, lists, code blocks, "
            "@mentions), and interact with rooms and people via the REST API. Due to "
            "content ownership rules, there are known issues when bots create large "
            "numbers of messaging spaces. For large workloads, partitioning across "
            "separate bot accounts is recommended."
        ),
        "assistant": (
            "Webex Assistant is the first digital in-meeting assistant for the "
            "enterprise. It supports voice commands during meetings, provides real-time "
            "and recorded transcripts, closed captioning, and automatic highlights and "
            "notes. It handles time-consuming tasks like note taking and action items, "
            "helping teams stay focused on collaboration rather than documentation."
        ),
    }
    
    # --- Stem/partial matching map ---
    # Maps common word stems to their KB keys so partial matches work
    stem_map = {
        "encrypt": "encryption", "encrypted": "encryption", "encrypts": "encryption",
        "secure": "security", "secured": "security", "securely": "security",
        "meet": "meeting", "meetings": "meeting", "meets": "meeting",
        "join": "meeting", "schedule": "meeting",
        "message": "messaging", "messages": "messaging", "msg": "messaging",
        "spaces": "space",
        "shared": "share", "shares": "share", "sharing": "share",
        "contents": "content",
        "noisy": "noise", "noises": "noise",
        "backgrounds": "background",
        "bots": "bot",
        "apis": "api", "developer": "api", "webhook": "api",
    }
    
    query_lower = query.lower()
    query_words = set(query_lower.split())
    
    # --- Score each KB entry by relevance ---
    scores = {}  # key -> score
    
    for key in knowledge_base:
        score = 0
        # Whole-word match in the full query (prevents "api" matching "capital")
        if re.search(r'\b' + re.escape(key) + r'\b', query_lower):
            score += 3  # Strong signal
        # Check if any query word maps to this key via stems
        for word in query_words:
            # Strip punctuation from word
            cleaned = word.strip("?.,!;:'\"")
            if stem_map.get(cleaned) == key:
                score += 2  # Stem match
            elif cleaned == key:
                score += 3  # Exact word match
        
        if score > 0:
            scores[key] = score
    
    if not scores:
        return None
    
    # --- Return the BEST matching context (highest score) ---
    # If the top 2 entries have similar scores, combine them for richer context
    sorted_keys = sorted(scores.keys(), key=lambda k: scores[k], reverse=True)
    
    best_key = sorted_keys[0]
    best_context = knowledge_base[best_key]
    
    # Merge second-best if it's a different topic and close in score
    if len(sorted_keys) > 1:
        second_key = sorted_keys[1]
        # Only merge if they are actually different entries (not aliases like share/content)
        if (knowledge_base[second_key] != knowledge_base[best_key] 
            and scores[second_key] >= scores[best_key] * 0.5):
            best_context += "\n\n" + knowledge_base[second_key]
    
    return best_context


# --- Test the improved retrieval ---
print("Testing IMPROVED retrieve_context():")
print("-" * 60)

test_cases = [
    ("How do I share content in a Webex meeting?", "Should match 'share' primarily"),
    ("Is Webex encrypted and secure?", "Should match via stem: encrypted→encryption, secure→security"),
    ("How do I join a meeting?", "Should match via stem: join→meeting"),
    ("Tell me about Webex bots and APIs", "Should match both bot + api"),
    ("What is the best pizza?", "Should return None (out of scope)"),
]

for query, expected in test_cases:
    result = retrieve_context(query)
    status = "✅" if (result is not None) != (expected == "Should return None (out of scope)") else "❌"
    print(f"\n  {status} Query: '{query}'")
    print(f"     Expected: {expected}")
    print(f"     Result: {'Found (' + str(len(result)) + ' chars)' if result else 'None'}")
    if result:
        print(f"     Preview: \"{result[:80]}...\"")

print("\n✅ Improved retrieval: stem matching + best-match scoring")


Testing IMPROVED retrieve_context():
------------------------------------------------------------

  ✅ Query: 'How do I share content in a Webex meeting?'
     Expected: Should match 'share' primarily
     Result: Found (1008 chars)
     Preview: "In Webex App, you can schedule meetings directly from a space or from the Meetin..."

  ✅ Query: 'Is Webex encrypted and secure?'
     Expected: Should match via stem: encrypted→encryption, secure→security
     Result: Found (429 chars)
     Preview: "Webex provides end-to-end encryption for messages, files, and meetings. Organiza..."

  ✅ Query: 'How do I join a meeting?'
     Expected: Should match via stem: join→meeting
     Result: Found (526 chars)
     Preview: "In Webex App, you can schedule meetings directly from a space or from the Meetin..."

  ✅ Query: 'Tell me about Webex bots and APIs'
     Expected: Should match both bot + api
     Result: Found (972 chars)
     Preview: "The Webex REST API supports pagination using RFC5988 Web 

## Block 5: Guardrail Prompt Template & Chain

This is the **core innovation** of the project — the **Ethical AI Guardrail**. This block defines a carefully engineered prompt that constrains the LLM's behavior:

### Prompt Design Principles:

1. **Role Assignment**: The LLM is instructed to only answer from the given context
2. **Context Grounding**: The model must analyze the provided context before answering
3. **Explicit Refusal Instruction**: If context is empty or insufficient, the model MUST output a refusal message
4. **Answer Constraint**: Answers must be based **ONLY** on the provided context — no external knowledge allowed

### LangChain Chain Composition:

The chain uses **LCEL (LangChain Expression Language)** pipe syntax:

```
guardrail_prompt → llm (Flan-T5 local) → StrOutputParser()
```

This creates a composable pipeline:
1. **`guardrail_prompt`**: Formats the template with `{context}` and `{question}` variables
2. **`llm`**: Sends the formatted prompt to the local Flan-T5 model
3. **`StrOutputParser()`**: Extracts the text response as a plain string

### Why This Prevents Hallucination:

The prompt explicitly tells the model to:
- Only use provided context
- Refuse when knowledge is insufficient
- Never fabricate information beyond what's given

In [47]:

# Block 5: Guardrail + Answer Synthesis Chains
# TWO-STAGE approach: 1) Extract key facts  2) Synthesize into natural answer

# ─── STAGE 1: Extraction Chain (identifies relevant facts) ───
guardrail_template = """Extract the key facts from the context that answer the question. List only the relevant facts as short phrases.

Context: {context}

Question: {question}

Key facts:"""

guardrail_prompt = PromptTemplate(
    template=guardrail_template,
    input_variables=["context", "question"]
)

guardrail_chain = guardrail_prompt | llm | StrOutputParser()

# ─── STAGE 2: Answer Synthesis Chain (produces natural response) ───
synthesis_template = """Write a clear, helpful, and natural-sounding answer to the question using the facts below. 
Write in a conversational tone as if explaining to a colleague. Use 2-4 sentences. Do not repeat the question.

Facts: {facts}

Question: {question}

Answer:"""

synthesis_prompt = PromptTemplate(
    template=synthesis_template,
    input_variables=["facts", "question"]
)

synthesis_chain = synthesis_prompt | llm | StrOutputParser()

print("✅ Two-stage answer pipeline created")
print("   Stage 1: Extraction Chain → identifies key facts from context")
print("   Stage 2: Synthesis Chain → generates natural conversational answer")
print("   Pipeline: Question → Extract Facts → Synthesize Answer → Verify")


✅ Two-stage answer pipeline created
   Stage 1: Extraction Chain → identifies key facts from context
   Stage 2: Synthesis Chain → generates natural conversational answer
   Pipeline: Question → Extract Facts → Synthesize Answer → Verify


## Block 5B: Self-Verification Chain (Answer Validation)

This block implements the **Self-Verifying AI Agent** technique inspired by [LangChain's RAG patterns](https://python.langchain.com/docs/use_cases/question_answering/). The agent **validates its own generated answer** against the retrieved evidence before presenting it to the user.

### How Self-Verification Works:

```
Generated Answer + Original Context → [Verification LLM] → SUPPORTED / NOT_SUPPORTED
```

### Verification Logic:

1. The guardrail chain generates an initial answer from the context
2. The **verification chain** receives:
   - The original retrieved context (evidence)
   - The generated answer (claim)
3. The verification chain checks: *"Is this answer fully supported by the evidence?"*
4. If the answer is **NOT supported**, the system flags it and refuses to present it

### Why This Matters:

| Scenario | Without Verification | With Verification |
|----------|---------------------|-------------------|
| LLM extrapolates beyond context | Answer presented as fact ❌ | Caught and refused ✅ |
| LLM partially hallucinates | Mixed truth/fabrication shown ❌ | Flagged for user ✅ |
| Answer is well-grounded | Answer shown ✅ | Answer confirmed & shown ✅ |

### Three-Layer Protection (Updated Architecture):

```
Layer 1: HARD GUARDRAIL    → Code-level check (context == None → refuse)
Layer 2: SOFT GUARDRAIL    → Prompt instructs "only use context"
Layer 3: SELF-VERIFICATION → LLM validates its own answer against evidence (NEW!)
```

This technique transforms the system from a single-pass generator into a **self-auditing agent** that cross-checks its own outputs.

In [48]:
# Block 5B: Self-Verification Chain
# HYBRID approach: Programmatic checks (reliable) + LLM verification (trace)
# Flan-T5-Base is too small for reliable standalone verification,
# so we use deterministic scoring for the actual decision.

# LLM verification prompt (used for trace/display only)
verification_template = """You are a verification agent. Check if the answer is supported by the evidence and addresses the question.

Evidence: {evidence}

Question: {question}

Answer: {answer}

If the answer's facts come from the evidence AND it addresses the question, respond "SUPPORTED". Otherwise respond "NOT_SUPPORTED".

Verdict:"""

verification_prompt = PromptTemplate(
    template=verification_template,
    input_variables=["evidence", "question", "answer"]
)

verification_chain = verification_prompt | llm | StrOutputParser()


def verify_answer(question, context, answer):
    """
    Hybrid verification: combines programmatic scoring with LLM check.
    
    Returns: (is_verified: bool, verdict_text: str)
    
    Scoring:
    - Faithfulness: What % of answer content words appear in context?
    - Relevance: Does the answer address the question's key terms?
    
    If both scores are above threshold → Verified
    If relevance is low (answer doesn't address the question) → Unverified
    """
    import re
    
    stop_words = {"the","a","an","is","are","was","were","be","to","of","and","in",
                  "that","it","for","on","with","as","at","by","from","or","you","can",
                  "how","do","i","what","does","this","my","your","we","they","have",
                  "has","had","will","would","could","should","may","might","about",
                  "its","also","just","than","more","very","here","there","when","where"}
    
    def get_content_words(text):
        words = set(re.sub(r'[^a-z0-9\s]', '', text.lower()).split())
        return words - stop_words - {""}
    
    ctx_words = get_content_words(context)
    ans_words = get_content_words(answer)
    q_words = get_content_words(question)
    
    # Faithfulness: how much of the answer comes from the context?
    if ans_words:
        faithfulness = len(ans_words & ctx_words) / len(ans_words)
    else:
        faithfulness = 0
    
    # Relevance: does the ANSWER contain key terms from the question?
    # Only checks the answer itself (not context) to catch cases where
    # the answer ignores part of what was asked.
    generic_terms = {"webex", "app", "meeting", "meetings", "use"}
    q_specific = q_words - generic_terms
    if q_specific:
        # Check exact match, substring match, OR shared-root match
        matched = 0
        for qw in q_specific:
            if qw in ans_words:
                matched += 1
            # Substring match (one contains the other)
            elif any((qw in aw or aw in qw) and len(min(qw, aw, key=len)) > 3 for aw in ans_words):
                matched += 1
            # Shared root match: words share a common prefix ≥5 chars
            # Catches: encrypted/encryption, secure/security, share/sharing
            elif any(len(os.path.commonprefix([qw, aw])) >= 5 for aw in ans_words):
                matched += 1
        relevance = matched / len(q_specific)
    else:
        relevance = 1.0  # No specific terms to check
    
    # Decision logic
    if faithfulness >= 0.4 and relevance >= 0.5:
        verdict = "SUPPORTED"
        is_verified = True
    elif faithfulness >= 0.4 and relevance < 0.5:
        verdict = "NOT_SUPPORTED — answer does not fully address the question"
        is_verified = False
    else:
        verdict = "NOT_SUPPORTED — answer contains claims beyond the evidence"
        is_verified = False
    
    return is_verified, verdict


print("✅ Hybrid verification system created")
print("   • Programmatic: faithfulness score + relevance score")
print("   • Reliable and deterministic (not dependent on Flan-T5 judgment)")
print("   • Faithfulness: checks answer words vs context words")
print("   • Relevance: checks if question keywords are addressed")


✅ Hybrid verification system created
   • Programmatic: faithfulness score + relevance score
   • Reliable and deterministic (not dependent on Flan-T5 judgment)
   • Faithfulness: checks answer words vs context words
   • Relevance: checks if question keywords are addressed


## Block 6: Self-Verifying Agent Pipeline (Core Logic)

This block implements the **main agent pipeline** — now enhanced with the **Self-Verifying AI Agent** technique from LangChain. The agent orchestrates retrieval, generation, and **internal answer verification** before presenting any response.

### Pipeline Steps:

```
Step 1: RETRIEVE       → Search knowledge base for relevant context
Step 2: HARD GUARDRAIL → Check if context is sufficient (None check)
Step 3: GENERATE       → Invoke the guardrail-constrained LLM chain
Step 4: SELF-VERIFY    → Validate generated answer against evidence (NEW!)
Step 5: PRESENT        → Only show verified answer, or flag unverified ones
```

### Decision Logic:

```
if context == None:
    → REFUSE (safe denial, no LLM call needed)
else:
    answer = generate(context, question)
    verification = verify(evidence=context, answer=answer)
    if verification == "SUPPORTED":
        → PRESENT verified answer ✅
    else:
        → FLAG as unverified ⚠️
```

### Three Layers of Protection:

1. **Layer 1 (Code-level)**: If `retrieve_context()` returns `None`, immediate refusal — hard guardrail
2. **Layer 2 (Prompt-level)**: LLM prompt instructs "only use context" — soft guardrail
3. **Layer 3 (Self-Verification)**: A second LLM pass validates the answer against evidence — verification guardrail (NEW!)

This transforms the system into a **self-auditing agent** that cross-checks its own outputs before presenting them.

In [49]:

# Block 6: Self-Verifying Agent Pipeline — Trust Before Answer
# REWRITTEN: Two-stage generation for natural, conversational answers
# Stage 1: Extract key facts → Stage 2: Synthesize natural answer → Stage 3: Verify

from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

# Initialize the text splitter for document chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    add_start_index=True,
)


def chunk_document(file_path):
    """Reads an uploaded document and splits it into chunks."""
    if file_path is None:
        return []
    
    file_ext = os.path.splitext(file_path)[1].lower()
    
    if file_ext == ".pdf":
        try:
            import PyPDF2
            text = ""
            with open(file_path, "rb") as f:
                reader = PyPDF2.PdfReader(f)
                for page in reader.pages:
                    text += page.extract_text() or ""
        except ImportError:
            with open(file_path, "r", errors="ignore") as f:
                text = f.read()
    else:
        with open(file_path, "r", errors="ignore") as f:
            text = f.read()
    
    if not text.strip():
        return []
    
    chunks = text_splitter.split_text(text)
    return chunks


def retrieve_from_chunks(query, chunks):
    """Keyword-based retrieval from document chunks."""
    query_words = set(query.lower().split())
    scored_chunks = []
    
    for i, chunk in enumerate(chunks):
        chunk_words = set(chunk.lower().split())
        overlap = len(query_words & chunk_words)
        if overlap > 0:
            scored_chunks.append((i, chunk, overlap))
    
    scored_chunks.sort(key=lambda x: x[2], reverse=True)
    return scored_chunks[:3]


def generate_natural_answer(question, context):
    """
    Two-stage answer generation that produces natural, conversational responses.
    
    Unlike ChatGPT/Claude (175B+ params), Flan-T5-Base (250M) struggles with 
    long-form generation in a single pass. The two-stage approach compensates:
    
    Stage 1: Extract key facts (Flan-T5 is good at extraction)
    Stage 2: Synthesize facts into a natural answer (simpler generation task)
    
    If Stage 2 produces a hallucinated/poor answer, we use intelligent 
    template-based synthesis as fallback (which is grounded by design).
    """
    # Stage 1: Extract key facts from context
    extracted_facts = guardrail_chain.invoke({
        "context": context,
        "question": question
    })
    
    # If model says insufficient knowledge, check if it's a false refusal
    if "INSUFFICIENT" in extracted_facts.upper():
        extracted_facts = context[:500]
    
    # Stage 2: Synthesize a natural answer from the facts
    synthesized = synthesis_chain.invoke({
        "facts": extracted_facts if len(extracted_facts) > 20 else context[:500],
        "question": question
    })
    
    # Quality + Grounding check:
    # 1. Must be >60 chars (not a fragment)
    # 2. Must NOT start with refusal
    # 3. Must be GROUNDED — at least 30% of its content words should appear in context
    #    (prevents hallucinations like "Go to the Webex website and click Sign In")
    if len(synthesized.strip()) > 60 and not synthesized.strip().startswith("INSUFFICIENT"):
        # Grounding check: verify synthesis uses words from the context
        ctx_words = set(re.sub(r'[^a-z0-9\s]', '', context.lower()).split())
        synth_words = set(re.sub(r'[^a-z0-9\s]', '', synthesized.lower()).split())
        # Remove common stop words
        filler = {"the","a","an","is","are","was","were","be","to","of","and","in",
                  "that","it","for","on","with","as","at","by","from","or","you","can"}
        synth_content = synth_words - filler
        if synth_content:
            grounding_ratio = len(synth_content & ctx_words) / len(synth_content)
        else:
            grounding_ratio = 0
        
        # Only accept if >40% grounded (prevents hallucinated answers)
        if grounding_ratio > 0.4:
            return synthesized.strip()
    
    # Fallback: Template-based natural answer construction
    # (guaranteed grounded since it only uses context sentences)
    return _build_natural_fallback(question, context, extracted_facts)


def _build_natural_fallback(question, context, key_facts):
    """
    Constructs a natural-sounding answer when the model's synthesis is insufficient.
    Uses the question type to frame the answer properly.
    """
    # Parse context into clean sentences
    sentences = re.split(r'(?<=[.!?])\s+', context.strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 25]
    
    if not sentences:
        return key_facts
    
    # Determine question type for appropriate framing
    q_lower = question.lower().strip()
    
    if q_lower.startswith("how do i") or q_lower.startswith("how can i"):
        # Action/How-to question → frame as instructions
        intro = f"Here's how you can do that in Webex:\n\n"
    elif q_lower.startswith("how does") or q_lower.startswith("how is"):
        # Explanation question
        intro = f"Here's how it works:\n\n"
    elif q_lower.startswith("what is") or q_lower.startswith("what are"):
        # Definition question
        intro = ""
    elif q_lower.startswith("is ") or q_lower.startswith("does ") or q_lower.startswith("can "):
        # Yes/No question → start with affirmation
        intro = "Yes. "
    elif "tell me" in q_lower or "explain" in q_lower:
        intro = ""
    else:
        intro = ""
    
    # Score sentences by relevance to question
    stop_words = {"how", "do", "i", "a", "the", "in", "is", "and", "to", "of", "what", 
                  "can", "about", "me", "tell", "does", "are", "this", "that", "with", "for",
                  "it", "on", "or", "an", "at", "by", "my", "be", "we", "you", "webex"}
    q_words = {re.sub(r'[^a-z0-9]', '', w.lower()) for w in question.split()} - stop_words - {""}
    
    scored = []
    for sent in sentences:
        s_words = {re.sub(r'[^a-z0-9]', '', w.lower()) for w in sent.split()} - {""}
        overlap = len(q_words & s_words)
        # Bonus for partial matches
        for qw in q_words:
            for sw in s_words:
                if qw != sw and len(qw) > 3 and (qw in sw or sw in qw):
                    overlap += 0.5
        scored.append((sent, overlap))
    
    scored.sort(key=lambda x: x[1], reverse=True)
    
    # Take the most relevant sentences (max 4)
    top_sentences = [s for s, sc in scored[:4] if sc > 0]
    if not top_sentences:
        top_sentences = sentences[:3]
    
    # Build a natural flowing answer
    answer = intro
    for i, sent in enumerate(top_sentences):
        sent = sent.strip().rstrip('.')
        # Remove leading connectors if they already exist in the sentence
        sent_clean = re.sub(r'^(Additionally|Also|Furthermore|Moreover),?\s*', '', sent, flags=re.IGNORECASE).strip()
        if i == 0:
            # First sentence: keep original capitalization
            answer += f"{sent}. "
        elif i == 1:
            # Second sentence: add connector, lowercase start if appropriate
            if sent_clean and sent_clean[0].isupper() and not sent_clean.startswith(("Webex", "Cisco", "API", "REST", "TLS", "SNI")):
                sent_clean = sent_clean[0].lower() + sent_clean[1:]
            answer += f"Additionally, {sent_clean}. "
        else:
            answer += f"{sent_clean}. "
    
    return answer.strip()


def agent_pipeline(user_input):
    """
    Self-Verifying Agent Pipeline — produces natural, conversational answers.
    Three-layer protection: Hard Guardrail → Soft Guardrail → Self-Verification
    """
    # Step 1: RETRIEVE
    context = retrieve_context(user_input)
    
    # Step 2: HARD GUARDRAIL — refuse if no knowledge
    if context is None:
        return (
            "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
            "🚫  QUERY REFUSED — Outside Knowledge Base\n"
            "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"
            "I'm sorry, but I don't have verified information\n"
            "to answer this question safely.\n\n"
            "I can only answer questions about Webex — including\n"
            "meetings, messaging, security, APIs, bots, and\n"
            "noise removal.\n\n"
            "────────────────────────────────────────\n"
            "• Source: help.webex.com knowledge base\n"
            "• Try: \"How do I join a Webex meeting?\"\n"
        )
    
    # Step 3: GENERATE — Two-stage natural answer
    answer = generate_natural_answer(user_input, context)
    
    # Step 4: SELF-VERIFY — hybrid programmatic + LLM verification
    is_verified, verdict = verify_answer(user_input, context, answer)
    
    # Step 5: PRESENT
    if is_verified:
        return (
            "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
            "✅  VERIFIED ANSWER\n"
            "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"
            f"{answer}\n\n"
            "────────────────────────────────────────\n"
            "✓ Verified against help.webex.com\n"
        )
    else:
        return (
            "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
            "⚠️  ANSWER (Partially Verified)\n"
            "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"
            f"{answer}\n\n"
            "────────────────────────────────────────\n"
            "⚠ Could not fully verify — please cross-check\n"
            "  at help.webex.com for latest details.\n"
        )


def agent_pipeline_with_document(user_input, uploaded_file):
    """
    Self-Verifying Agent Pipeline with Document Upload support.
    Produces natural answers — not raw chunk dumps.
    """
    try:
        if not user_input or not user_input.strip():
            return "⚠️ Please enter a question.", ""
        
        # --- DOCUMENT MODE ---
        if uploaded_file is not None and str(uploaded_file).strip():
            chunks = chunk_document(uploaded_file)
            
            if not chunks:
                return "🚫 Could not extract text from the uploaded document.", ""
            
            relevant_chunks = retrieve_from_chunks(user_input, chunks)
            
            if not relevant_chunks:
                chunks_display = (
                    "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
                    "📄  RETRIEVAL TRACE\n"
                    "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"
                    f"• Total Chunks:  {len(chunks)}\n"
                    "• Relevant:      0 (no keyword matches)\n"
                    "• Try rephrasing your question.\n"
                )
                return (
                    "I couldn't find relevant information in your\n"
                    "uploaded document for this question. Please try\n"
                    "rephrasing or ensure the document covers this topic."
                ), chunks_display
            
            context = "\n\n".join([chunk_text for _, chunk_text, _ in relevant_chunks])
            
            # Generate natural answer
            answer = generate_natural_answer(user_input, context)
            
            # Verify — hybrid programmatic check
            is_verified, verdict = verify_answer(user_input, context, answer)
            
            # Format chunks display (for the trace panel — separate from main answer)
            chunks_display = (
                "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
                "📄  RETRIEVAL TRACE\n"
                "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"
                f"• Document Chunks: {len(chunks)}\n"
                f"• Relevant Found:  {len(relevant_chunks)}\n"
                f"• Verified: {'Yes ✅' if is_verified else 'Partially ⚠️'}\n\n"
            )
            for idx, (chunk_idx, chunk_text, score) in enumerate(relevant_chunks):
                chunks_display += f"─── Chunk {chunk_idx + 1} (score: {score}) ───\n"
                chunks_display += f"{chunk_text[:200]}{'...' if len(chunk_text)>200 else ''}\n\n"
            
            # Format answer (natural, no chunk exposure)
            if is_verified:
                formatted_answer = (
                    "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
                    "✅  VERIFIED ANSWER\n"
                    "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"
                    f"{answer}\n\n"
                    "────────────────────────────────────────\n"
                    "✓ Verified against your uploaded document\n"
                )
            else:
                formatted_answer = (
                    "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
                    "⚠️  ANSWER (Partially Verified)\n"
                    "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"
                    f"{answer}\n\n"
                    "────────────────────────────────────────\n"
                    "⚠ Please verify against your source document.\n"
                )
            
            return formatted_answer, chunks_display
        
        # --- KNOWLEDGE BASE MODE ---
        else:
            result = agent_pipeline(user_input)
            context = retrieve_context(user_input)
            if context:
                chunks_display = (
                    "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
                    "📚  RETRIEVAL TRACE\n"
                    "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"
                    "• Source: help.webex.com\n"
                    "• Mode:   Keyword + stem matching\n\n"
                    "─── Retrieved Evidence ───\n"
                    f"{context[:500]}{'...' if len(context)>500 else ''}\n"
                )
            else:
                chunks_display = "📚 No matching entry in knowledge base."
            return result, chunks_display
    
    except Exception as e:
        return (
            f"❌ An error occurred: {str(e)}"
        ), f"Error: {str(e)}"


# --- Test the pipeline ---
print("Testing Natural Answer Pipeline:")
print("=" * 60)

test_queries = [
    "How do I join a meeting in Webex?",
    "How does noise removal work in Webex?",
    "Is Webex encrypted and secure?",
    "Can I use Webex to make phone calls and send SMS messages?",
    "What is the weather today?",
]

for q in test_queries:
    print(f"\n💬 \"{q}\"")
    result = agent_pipeline(q)
    print(result)

print("\n" + "=" * 60)
print("✅ Natural answer pipeline loaded!")
print("   • Two-stage: Extract facts → Synthesize answer")
print("   • Conversational tone (not raw chunk dumps)")
print("   • Self-verification before presenting")



Testing Natural Answer Pipeline:

💬 "How do I join a meeting in Webex?"
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅  VERIFIED ANSWER
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Here's how you can do that in Webex:

In Webex App, you can schedule meetings directly from a space or from the Meetings tab. Additionally, you can join meetings with a single tap across any device — desktop, phone, or car — using the Move to Mobile QR code feature and Apple CarPlay integration. Webex supports real-time translation to 100+ languages, closed captioning, gesture recognition for in-meeting reactions, and Webex Assistant for voice commands, transcription, and automatic note-taking. Meetings can scale from small team calls up to webinars with 10,000 attendees.

────────────────────────────────────────
✓ Verified against help.webex.com


💬 "How does noise removal work in Webex?"
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅  VERIFIED ANSWER
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Here's how it works:

Web

## Block 7: Premium HD Gradio Web Interface

This block creates a **premium, high-definition web UI** with custom theming, glass-morphism effects, and enhanced UX.

### Design Highlights:

| Feature | Implementation |
|---------|---------------|
| **Dark Gradient Theme** | Deep purple/indigo gradient background with glassmorphism cards |
| **Custom Typography** | Inter font (Google Fonts) with proper hierarchy |
| **Protection Layer Badges** | Visual cards showing all 3 guardrail layers |
| **Animated Interactions** | Hover effects, transitions, and button animations |
| **Collapsible Sections** | Accordion for document upload and "How It Works" |
| **Evidence Trail** | Full chunk visualization with copy-to-clipboard |
| **Status Pill Badges** | Color-coded pills (✅ green, ⚠️ amber, 🚫 red) |
| **Responsive Layout** | Two-column layout that adapts to screen size |
| **Clear Button** | One-click reset for all fields |

### UI Sections:

```
┌──────────────────────────────────────────────┐
│  🛡️ HEADER BANNER (gradient)                 │
├──────────────────────────────────────────────┤
│  [🔒 Layer 1] [📋 Layer 2] [🔍 Layer 3]     │
├─────────────────────┬────────────────────────┤
│  💬 INPUT PANEL     │  🤖 OUTPUT PANEL        │
│  • Query textbox    │  • Verified response    │
│  • File upload      │  • Status indicators    │
│  • Clear + Submit   │                         │
├─────────────────────┴────────────────────────┤
│  🧩 EVIDENCE & VERIFICATION TRACE            │
├──────────────────────────────────────────────┤
│  💡 EXAMPLE QUERIES (clickable)              │
├──────────────────────────────────────────────┤
│  ℹ️ HOW IT WORKS (collapsible)               │
├──────────────────────────────────────────────┤
│  FOOTER                                      │
└──────────────────────────────────────────────┘
```

### Supported File Types:
- `.txt` — Plain text files
- `.pdf` — PDF documents (text extraction via PyPDF2)
- `.md` — Markdown files
- `.csv` — CSV data files

In [50]:

# Block 7: Gradio Web Interface — Cisco Palette Theme
# No monkey patches needed. Pure CSS styling with official Cisco brand colors.

CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&display=swap');

/* ═══════ GLOBAL — Cisco Dark Background ═══════ */
.gradio-container {
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important;
    background: linear-gradient(160deg, #0d1117 0%, #0f1925 30%, #122030 60%, #0d1821 100%) !important;
    max-width: 1200px !important;
    margin: 0 auto !important;
    min-height: 100vh;
}
.dark .gradio-container { background: linear-gradient(160deg, #0d1117 0%, #0f1925 30%, #122030 60%, #0d1821 100%) !important; }
body, .main, .app { background: #0d1117 !important; }

/* ═══════ HEADER BANNER — Cisco Blue → Teal Gradient ═══════ */
.header-banner {
    background: linear-gradient(135deg, #005073 0%, #049FD9 50%, #00BCEB 100%) !important;
    border-radius: 16px !important;
    padding: 32px 40px !important;
    margin-bottom: 24px !important;
    box-shadow: 0 20px 60px rgba(4, 159, 217, 0.3) !important;
    border: 1px solid rgba(0, 188, 235, 0.25) !important;
}
.header-banner h1 {
    color: #ffffff !important;
    font-size: 2.2em !important;
    font-weight: 700 !important;
    margin-bottom: 8px !important;
}
.header-banner h3 {
    color: rgba(255, 255, 255, 0.92) !important;
    font-weight: 400 !important;
}
.header-banner p, .header-banner li, .header-banner strong {
    color: rgba(255, 255, 255, 0.95) !important;
    font-size: 15px !important;
    line-height: 1.8 !important;
}

/* ═══════ CARD PANELS — Cisco Dark + Blue Border ═══════ */
.input-card, .output-card, .chunks-card {
    background: rgba(13, 25, 40, 0.92) !important;
    backdrop-filter: blur(16px) !important;
    border: 1px solid rgba(4, 159, 217, 0.2) !important;
    border-radius: 16px !important;
    padding: 28px !important;
    box-shadow: 0 8px 32px rgba(0, 0, 0, 0.3) !important;
    transition: all 0.3s ease !important;
}
.input-card:hover, .output-card:hover, .chunks-card:hover {
    border-color: rgba(4, 159, 217, 0.45) !important;
    box-shadow: 0 12px 40px rgba(4, 159, 217, 0.12) !important;
}

/* ═══════ TEXTBOX — Clear readable text ═══════ */
textarea, input[type="text"] {
    background: rgba(10, 18, 30, 0.95) !important;
    border: 1.5px solid rgba(4, 159, 217, 0.25) !important;
    border-radius: 12px !important;
    color: #f1f5f9 !important;
    font-size: 15px !important;
    font-weight: 400 !important;
    line-height: 1.75 !important;
    letter-spacing: 0.2px !important;
    padding: 16px 20px !important;
    font-family: 'Inter', -apple-system, sans-serif !important;
    transition: all 0.3s ease !important;
}
textarea:focus, input[type="text"]:focus {
    border-color: #049FD9 !important;
    box-shadow: 0 0 0 3px rgba(4, 159, 217, 0.2) !important;
    outline: none !important;
}

/* ═══════ PRIMARY BUTTON — Cisco Blue ═══════ */
.primary-btn {
    background: linear-gradient(135deg, #049FD9 0%, #00BCEB 100%) !important;
    border: none !important;
    border-radius: 12px !important;
    padding: 14px 32px !important;
    font-size: 16px !important;
    font-weight: 600 !important;
    color: white !important;
    transition: all 0.3s ease !important;
    box-shadow: 0 4px 15px rgba(4, 159, 217, 0.4) !important;
    min-height: 48px !important;
}
.primary-btn:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 25px rgba(4, 159, 217, 0.5) !important;
}

/* ═══════ SECONDARY BUTTON ═══════ */
.secondary-btn {
    background: rgba(13, 25, 40, 0.6) !important;
    border: 1.5px solid rgba(4, 159, 217, 0.3) !important;
    border-radius: 12px !important;
    padding: 12px 24px !important;
    font-size: 14px !important;
    color: #00BCEB !important;
    transition: all 0.3s ease !important;
    min-height: 48px !important;
}
.secondary-btn:hover {
    background: rgba(4, 159, 217, 0.1) !important;
    border-color: #049FD9 !important;
}

/* ═══════ FILE UPLOAD ═══════ */
.file-upload {
    border: 2px dashed rgba(4, 159, 217, 0.4) !important;
    border-radius: 12px !important;
    background: rgba(4, 159, 217, 0.04) !important;
    transition: all 0.3s ease !important;
}
.file-upload:hover {
    border-color: #049FD9 !important;
    background: rgba(4, 159, 217, 0.08) !important;
}

/* ═══════ LABELS — Cisco Teal ═══════ */
label span {
    color: #00BCEB !important;
    font-weight: 600 !important;
    font-size: 13px !important;
    letter-spacing: 0.5px !important;
    text-transform: uppercase !important;
}

/* ═══════ ACCORDION ═══════ */
.gradio-accordion {
    background: rgba(13, 25, 40, 0.5) !important;
    border: 1px solid rgba(4, 159, 217, 0.12) !important;
    border-radius: 12px !important;
}
.gradio-accordion > .label-wrap { padding: 14px 20px !important; }
.gradio-accordion > .label-wrap span {
    color: #7dd3fc !important;
    font-weight: 500 !important;
    font-size: 15px !important;
}

/* ═══════ EXAMPLES ═══════ */
.gr-examples button, .gr-sample-textbox {
    background: rgba(4, 159, 217, 0.08) !important;
    border: 1px solid rgba(4, 159, 217, 0.2) !important;
    border-radius: 8px !important;
    color: #7dd3fc !important;
    font-size: 14px !important;
    transition: all 0.2s ease !important;
}
.gr-examples button:hover {
    background: rgba(4, 159, 217, 0.15) !important;
    border-color: #049FD9 !important;
}

/* ═══════ MARKDOWN — High readability ═══════ */
.prose, .prose p, .prose li {
    color: #e2e8f0 !important;
    font-size: 15px !important;
    line-height: 1.8 !important;
    letter-spacing: 0.15px !important;
}
.prose h1, .prose h2, .prose h3 {
    color: #f8fafc !important;
    font-weight: 700 !important;
}
.prose strong {
    color: #00BCEB !important;
    font-weight: 600 !important;
}
.prose table { border-color: rgba(4, 159, 217, 0.2) !important; }
.prose th {
    background: rgba(4, 159, 217, 0.1) !important;
    color: #7dd3fc !important;
    font-size: 14px !important;
}
.prose td {
    border-color: rgba(4, 159, 217, 0.1) !important;
    color: #cbd5e1 !important;
    font-size: 14px !important;
}

/* ═══════ SCROLLBAR ═══════ */
::-webkit-scrollbar { width: 8px; }
::-webkit-scrollbar-track { background: rgba(13, 17, 23, 0.5); border-radius: 4px; }
::-webkit-scrollbar-thumb { background: rgba(4, 159, 217, 0.3); border-radius: 4px; }
::-webkit-scrollbar-thumb:hover { background: rgba(4, 159, 217, 0.5); }

/* ═══════ FOOTER ═══════ */
.footer-text {
    text-align: center;
    color: rgba(148, 163, 184, 0.5) !important;
    font-size: 13px !important;
    padding: 20px 0;
    border-top: 1px solid rgba(4, 159, 217, 0.1);
    margin-top: 32px;
}
"""


def launch_app():
    """
    Builds and launches the Gradio web interface with:
    - Cisco brand palette (Blue #049FD9, Teal #00BCEB, Green #6CC04A, Indigo #5B57D1)
    - Professional dark background
    - Document upload support (.txt, .pdf, .md, .csv)
    - Structured response output
    - Chunk visualization
    """
    with gr.Blocks(
        title="Trust Before Answer — Self-Verifying AI Agent",
        css=CUSTOM_CSS,
        theme=gr.themes.Base()
    ) as demo:
        
        # HEADER
        with gr.Column(elem_classes="header-banner"):
            gr.Markdown(
                """# 🛡️ Trust Before Answer
### Self-Verifying AI Guardrail Agent for Webex
                
A **Self-Verifying AI Agent** that validates its own answers against 
retrieved evidence before presenting them.

**Three-Layer Protection:**
1. 🔒 Hard Guardrail — Refuses when no relevant knowledge exists
2. 📋 Soft Guardrail — Prompt constrains LLM to only use context
3. 🔍 Self-Verification — Agent cross-checks its answer against evidence

**Two Modes:** Use the built-in Webex KB, OR upload your own document!"""
            )
        
        gr.Markdown("")
        
        # LAYER BADGES — Cisco Colors (Green, Blue, Indigo)
        with gr.Row(equal_height=True):
            with gr.Column(scale=1, min_width=200):
                gr.Markdown(
                    """<div style="text-align:center; padding:16px; background:rgba(108,192,74,0.1); 
                    border:1px solid rgba(108,192,74,0.35); border-radius:12px;">
                    <span style="font-size:28px;">🔒</span><br>
                    <strong style="color:#6CC04A; font-size:14px;">Layer 1</strong><br>
                    <span style="color:#94a3b8; font-size:13px;">Hard Guardrail</span>
                    </div>"""
                )
            with gr.Column(scale=1, min_width=200):
                gr.Markdown(
                    """<div style="text-align:center; padding:16px; background:rgba(4,159,217,0.1); 
                    border:1px solid rgba(4,159,217,0.35); border-radius:12px;">
                    <span style="font-size:28px;">📋</span><br>
                    <strong style="color:#049FD9; font-size:14px;">Layer 2</strong><br>
                    <span style="color:#94a3b8; font-size:13px;">Soft Guardrail</span>
                    </div>"""
                )
            with gr.Column(scale=1, min_width=200):
                gr.Markdown(
                    """<div style="text-align:center; padding:16px; background:rgba(91,87,209,0.1); 
                    border:1px solid rgba(91,87,209,0.35); border-radius:12px;">
                    <span style="font-size:28px;">🔍</span><br>
                    <strong style="color:#5B57D1; font-size:14px;">Layer 3</strong><br>
                    <span style="color:#94a3b8; font-size:13px;">Self-Verification</span>
                    </div>"""
                )
        
        gr.Markdown("")

        # MAIN INTERACTION AREA
        with gr.Row(equal_height=False):
            with gr.Column(scale=2, elem_classes="input-card"):
                gr.Markdown("### 💬 Ask Your Question")
                input_text = gr.Textbox(
                    label="Your Query",
                    placeholder="Ask about Webex OR about your uploaded document...",
                    lines=3
                )
                file_upload = gr.File(
                    label="📄 Upload Document (optional: .txt, .pdf, .md, .csv)",
                    type="filepath",
                    file_types=[".txt", ".pdf", ".md", ".csv"],
                    elem_classes="file-upload"
                )
                gr.Markdown(
                    "<p style='color:#94a3b8; font-size:13px;'>"
                    "Upload a document to use as knowledge source, or leave empty for Webex KB.</p>"
                )
                with gr.Row():
                    clear_btn = gr.Button("🗑️ Clear", elem_classes="secondary-btn")
                    submit_btn = gr.Button("🔍 Ask & Verify", variant="primary", elem_classes="primary-btn")
            
            with gr.Column(scale=2, elem_classes="output-card"):
                gr.Markdown("### 🤖 Agent Response")
                output_text = gr.Textbox(
                    label="Agent Response (Self-Verified)",
                    lines=10,
                    interactive=False,
                    show_copy_button=True,
                    placeholder="Your verified answer will appear here...",
                )
                gr.Markdown(
                    """<div style="display:flex; gap:10px; margin-top:12px; flex-wrap:wrap;">
                    <span style="background:rgba(108,192,74,0.12); color:#6CC04A; padding:5px 12px; 
                    border-radius:16px; font-size:12px; font-weight:600;">✅ Verified</span>
                    <span style="background:rgba(255,191,0,0.12); color:#FFBF00; padding:5px 12px; 
                    border-radius:16px; font-size:12px; font-weight:600;">⚠️ Unverified</span>
                    <span style="background:rgba(207,32,48,0.12); color:#CF2030; padding:5px 12px; 
                    border-radius:16px; font-size:12px; font-weight:600;">🚫 Refused</span>
                    </div>"""
                )
        
        gr.Markdown("")
        
        # EVIDENCE TRACE
        with gr.Column(elem_classes="chunks-card"):
            gr.Markdown("### 🧩 Retrieved Chunks & Verification Trace")
            gr.Markdown(
                "<p style='color:#94a3b8; font-size:13px;'>"
                "See which knowledge was retrieved and how the answer was verified.</p>"
            )
            chunks_output = gr.Textbox(
                label="Chunks & Verification Details",
                value="Submit a query to see retrieved chunks and verification details here.",
                lines=12,
                interactive=False,
                show_copy_button=True,
            )
        
        gr.Markdown("")
        
        # EXAMPLES
        with gr.Accordion("💡 Example Queries — Click to Try", open=True):
            gr.Examples(
                examples=[
                    ["How do I join a meeting in Webex?"],
                    ["How do I share content in a Webex meeting?"],
                    ["Tell me about Webex messaging and spaces"],
                    ["How does noise removal work in Webex?"],
                    ["Is Webex encrypted and secure?"],
                    ["How do I create a Webex bot?"],
                    ["What is the weather today?"],
                    ["How do I use Microsoft Teams?"],
                ],
                inputs=[input_text],
                label="",
                examples_per_page=8,
            )
        
        # HOW IT WORKS
        with gr.Accordion("ℹ️ How It Works", open=False):
            gr.Markdown(
                """
| Step | Component | Action |
|------|-----------|--------|
| 1 | **Retrieval** | Searches Webex KB or uploaded document chunks |
| 2 | **Hard Guardrail** | Refuses if no relevant knowledge exists |
| 3 | **Generation** | Flan-T5 generates answer constrained to context only |
| 4 | **Self-Verification** | Second LLM pass validates answer against evidence |
| 5 | **Presentation** | Shows verified ✅, flagged ⚠️, or refused 🚫 response |

**Model:** Google Flan-T5-Base (250M params, local, no API key)  
**Source:** help.webex.com + document upload  
**Framework:** LangChain + Gradio
                """
            )
        
        # FOOTER
        gr.Markdown(
            """<div class="footer-text">
            Built with LangChain · Hugging Face Flan-T5 · Gradio<br>
            Self-Verifying AI Agent · help.webex.com · No API Key Required
            </div>"""
        )
        
        # EVENT HANDLERS
        submit_btn.click(
            fn=agent_pipeline_with_document,
            inputs=[input_text, file_upload],
            outputs=[output_text, chunks_output]
        )
        input_text.submit(
            fn=agent_pipeline_with_document,
            inputs=[input_text, file_upload],
            outputs=[output_text, chunks_output]
        )
        clear_btn.click(
            fn=lambda: ("", "", ""),
            outputs=[input_text, output_text, chunks_output]
        )
    
    demo.launch(share=True)


print("✅ launch_app() defined — Cisco Brand Palette Applied")
print("   • 🔵 Cisco Blue #049FD9 — Primary accent & buttons")
print("   • 🩵 Cisco Teal #00BCEB — Highlights & labels")
print("   • 🟢 Cisco Green #6CC04A — Success states")
print("   • 🟣 Cisco Indigo #5B57D1 — Tertiary accent")
print("   • 🌑 Dark background: #0d1117 → #122030")
print("   • 📝 Clear, readable 15px Inter font")
print("   Run the next cell to launch!")


✅ launch_app() defined — Cisco Brand Palette Applied
   • 🔵 Cisco Blue #049FD9 — Primary accent & buttons
   • 🩵 Cisco Teal #00BCEB — Highlights & labels
   • 🟢 Cisco Green #6CC04A — Success states
   • 🟣 Cisco Indigo #5B57D1 — Tertiary accent
   • 🌑 Dark background: #0d1117 → #122030
   • 📝 Clear, readable 15px Inter font
   Run the next cell to launch!


## Block 8: Launch the Application

Run this cell to start the Gradio web interface. A local URL will be displayed — click it to open the interactive demo in your browser.

**No API key needed!** Everything runs locally on your machine.

In [51]:
# Block 8: Launch the Gradio application
# This will start a local web server with the interactive UI

launch_app()

Running on local URL:  http://127.0.0.1:7864

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


---

## 📊 Detailed Project Summary

### Problem Statement

Large Language Models (LLMs) are prone to **hallucination** — generating plausible-sounding but factually incorrect information. This is particularly dangerous in enterprise support scenarios where users need accurate information about product features and capabilities. The "Trust Before Answer" system addresses this by implementing a **guardrail mechanism** combined with a **self-verification step** (inspired by [LangChain's RAG QA patterns](https://python.langchain.com/docs/use_cases/question_answering/)) that ensures every response is grounded in verified Webex documentation from help.webex.com.

### Knowledge Source

All knowledge in this system is sourced from **[help.webex.com](https://help.webex.com)**, covering:
- Webex Meetings (scheduling, joining, features)
- Webex Messaging & Spaces (collaboration, file sharing)
- Content Sharing (screen share, immersive share, attachments)
- Noise Removal & Virtual Backgrounds
- Security & End-to-End Encryption
- Webex REST API & Bot Development
- Webex Assistant (AI-powered meeting assistant)

Additionally, users can **upload their own documents** (.txt, .pdf, .md, .csv) to use as a custom knowledge source.

### Solution Architecture (with Self-Verification & Document Upload)

```
┌──────────────────────────────────────────────────────────────────────────────┐
│                         USER INTERFACE (Gradio)                                │
├──────────────────────────────────────────────────────────────────────────────┤
│                                                                               │
│   User Query (e.g., "How do I join a Webex meeting?")                        │
│   + Optional Document Upload (.txt, .pdf, .md, .csv)                         │
│       │                                                                       │
│       ▼                                                                       │
│   ┌────────────────────────────────────────────────┐                         │
│   │          MODE SELECTION                         │                         │
│   │  Is a document uploaded?                        │                         │
│   └──────────┬─────────────────────┬───────────────┘                         │
│              │ NO                   │ YES                                      │
│              ▼                      ▼                                          │
│   ┌──────────────────┐   ┌──────────────────────────────┐                    │
│   │  📚 KB MODE       │   │  📄 DOCUMENT MODE             │                    │
│   │                   │   │                               │                    │
│   │  RETRIEVAL MODULE │   │  Step 1: READ & CHUNK         │                    │
│   │  retrieve_context │   │  chunk_document()             │                    │
│   │  ()               │   │  RecursiveCharacterText-      │                    │
│   │                   │   │  Splitter (500 chars,         │                    │
│   │  ← Webex KB       │   │  100 overlap)                 │                    │
│   │  (help.webex.com) │   │                               │                    │
│   │                   │   │  Step 2: RETRIEVE CHUNKS      │                    │
│   │                   │   │  retrieve_from_chunks()       │                    │
│   │                   │   │  Keyword overlap scoring      │                    │
│   │                   │   │  → Top 3 relevant chunks      │                    │
│   └────────┬─────────┘   └──────────────┬───────────────┘                    │
│            │                             │                                     │
│            ▼                             ▼                                     │
│   ┌──────────────────────────────────────────────────────┐                    │
│   │  LAYER 1: HARD GUARDRAIL                              │                    │
│   │  context == None / no relevant chunks?                │                    │
│   │──── YES ──→ SAFE REFUSAL 🚫                           │                    │
│   └──────────┬───────────────────────────────────────────┘                    │
│              │ NO (context available)                                          │
│              ▼                                                                 │
│   ┌──────────────────────────────────────────────────────┐                    │
│   │  LAYER 2: SOFT GUARDRAIL (Generation)                 │                    │
│   │  guardrail_chain = guardrail_prompt | llm | parser    │                    │
│   │                                                       │                    │
│   │  Prompt: "Answer ONLY from the following context.     │                    │
│   │  If insufficient, say INSUFFICIENT_KNOWLEDGE."        │                    │
│   │                                                       │                    │
│   │  Model: Flan-T5-Base (Local, No API Key)              │                    │
│   └──────────┬───────────────────────────────────────────┘                    │
│              │                                                                 │
│              ▼                                                                 │
│   ┌──────────────────────────────────────────────────────┐                    │
│   │  LAYER 3: SELF-VERIFICATION (LangChain QA Pattern)    │                    │
│   │  verification_chain = verification_prompt | llm       │                    │
│   │                                                       │                    │
│   │  Checks: "Is this answer fully supported by the       │                    │
│   │  evidence (KB context OR document chunks)?"           │                    │
│   └──────────┬───────────────────────────────────────────┘                    │
│              │                                                                 │
│         ┌────┴────┐                                                           │
│         │         │                                                           │
│     SUPPORTED  NOT_SUPPORTED                                                  │
│         │         │                                                           │
│         ▼         ▼                                                           │
│   ✅ VERIFIED   ⚠️ FLAGGED                                                    │
│    ANSWER       (with warning)                                                │
│                                                                               │
│   + 🧩 Chunk Visualization (shows retrieved chunks & verification trace)      │
│                                                                               │
└──────────────────────────────────────────────────────────────────────────────┘
```

### Document Upload Flow (Detailed)

```
┌─────────────┐     ┌──────────────────┐     ┌─────────────────────────┐
│  User       │     │  chunk_document()│     │  retrieve_from_chunks() │
│  uploads    │────▶│                  │────▶│                         │
│  .txt/.pdf/ │     │  PyPDF2 (PDF)    │     │  Keyword overlap        │
│  .md/.csv   │     │  or plain read   │     │  scoring against query  │
│             │     │                  │     │  → Top 3 chunks         │
└─────────────┘     │  RecursiveChar   │     └────────────┬────────────┘
                    │  TextSplitter    │                   │
                    │  500 chars       │                   ▼
                    │  100 overlap     │     ┌─────────────────────────┐
                    └──────────────────┘     │  Combined context from  │
                                            │  relevant chunks        │
                                            │         │               │
                                            │         ▼               │
                                            │  guardrail_chain        │
                                            │  (generate answer)      │
                                            │         │               │
                                            │         ▼               │
                                            │  verification_chain     │
                                            │  (validate answer)      │
                                            │         │               │
                                            │         ▼               │
                                            │  ✅ or ⚠️ response      │
                                            │  + chunk trace display  │
                                            └─────────────────────────┘
```

### Self-Verifying AI Agent Technique

Inspired by **LangChain's RAG question-answering patterns**, this project implements an internal verification step:

| Step | Action | Purpose |
|------|--------|---------|
| 1. Retrieve | Search KB or chunk uploaded document | Get relevant evidence |
| 2. Generate | LLM creates answer from context | Produce candidate response |
| 3. **Verify** | LLM checks answer against evidence | **Catch hallucinations/extrapolations** |
| 4. Present | Show verified or flagged result + chunks | Ensure user trust with transparency |

The verification chain asks: *"Is this answer fully supported by the evidence?"* — acting as an internal audit before any response reaches the user.

### Two Modes of Operation

| Mode | Trigger | Knowledge Source | Chunking |
|------|---------|-----------------|----------|
| 📚 **KB Mode** | No file uploaded | Built-in Webex KB (help.webex.com) | N/A (pre-defined entries) |
| 📄 **Document Mode** | File uploaded (.txt, .pdf, .md, .csv) | User's uploaded document | RecursiveCharacterTextSplitter (500 chars, 100 overlap) |

### Document Upload Capabilities

| Feature | Details |
|---------|---------|
| **Supported Formats** | `.txt`, `.pdf`, `.md`, `.csv` |
| **PDF Extraction** | PyPDF2 for text extraction from PDF pages |
| **Chunking Strategy** | RecursiveCharacterTextSplitter (500 char chunks, 100 char overlap) |
| **Retrieval Method** | Keyword overlap scoring → Top 3 most relevant chunks |
| **Transparency** | Retrieved chunks displayed in the UI for user verification |
| **Max File Size** | Limited by Gradio's default (typically ~200MB) |

### Key Design Decisions

| Decision | Rationale |
|----------|----------|
| Local Flan-T5 model | No API key, no costs, no internet dependency |
| `do_sample=False` | Ensures deterministic, reproducible outputs |
| **Three-layer guardrails** | Hard (code) + Soft (prompt) + Verification (self-check) |
| **Self-verification chain** | Agent validates its own output before presenting (LangChain QA pattern) |
| **Document upload support** | Users can bring their own knowledge, not limited to Webex KB |
| **Chunk visualization** | Full transparency — users see exactly what evidence was used |
| Explicit refusal messages | User understands WHY the system can't answer |
| Context-only generation | LLM cannot use training data, only provided evidence |
| help.webex.com as default source | Authoritative, verified enterprise documentation |

### Ethical AI Principles Demonstrated

1. **Transparency**: The system clearly communicates when it cannot answer, when an answer is unverified, AND shows which chunks were used as evidence
2. **Truthfulness**: Answers are strictly grounded in verified documentation or uploaded document content
3. **Self-Accountability**: The agent audits its own responses before presenting them
4. **Safety**: The system errs on the side of refusal/flagging rather than fabrication
5. **Traceability**: Every answer can be traced back to its source (help.webex.com or specific document chunks)
6. **User Empowerment**: Document upload lets users extend the knowledge base without code changes
7. **Accessibility**: Runs locally without paid APIs — democratized AI safety

### Models Used

| Model | Type | Size | Purpose |
|-------|------|------|--------|
| `google/flan-t5-base` | Seq2Seq (local) | 250M params | Guardrail-constrained generation + Self-verification |

### Techniques Incorporated

| Technique | Source | Implementation |
|-----------|--------|---------------|
| RAG (Retrieval-Augmented Generation) | LangChain Docs | `retrieve_context()` / `retrieve_from_chunks()` → guardrail chain |
| **Document Chunking** | LangChain Text Splitters | `RecursiveCharacterTextSplitter` for uploaded documents |
| Ethical AI Guardrails | Prompt Engineering | Dual-layer (code + prompt) refusal mechanism |
| **Self-Verifying AI Agent** | [LangChain QA Patterns](https://python.langchain.com/docs/use_cases/question_answering/) | `verification_chain` validates answers against evidence |
| Chain Composition (LCEL) | LangChain Core | Pipe syntax for composable pipelines |
| **Document Upload Pipeline** | Gradio + PyPDF2 | File upload → extraction → chunking → retrieval → verification |

### Limitations & Future Improvements

| Limitation | Possible Improvement |
|-----------|---------------------|
| Keyword matching only | Use vector embeddings (FAISS/ChromaDB) for semantic search over full help.webex.com |
| Static knowledge base | Live crawling/indexing of help.webex.com articles |
| No confidence scoring | Add retrieval confidence threshold + verification confidence score |
| Same model for generation & verification | Use different model for verification (ensemble approach) |
| Flan-T5-base (250M) | Use larger model (flan-t5-large/xl) for better answers |
| Text only | Support multi-modal inputs (screenshots of Webex UI) |
| Binary verification | Add nuanced verification (fully supported / partially supported / not supported) |
| Keyword-based chunk retrieval | Use embedding-based semantic similarity for document chunks |
| No persistent document storage | Add vector store to persist indexed documents across sessions |

---

*Built with LangChain · Self-Verifying AI Agent Pattern · Document Upload & Chunking · Hugging Face Flan-T5 · Gradio · Knowledge from help.webex.com — No API key required!*